# Image Quality vs Time of Night — LSSTCam

Scatter plots of ConsDB image-quality metrics as a function of **time of night**, split into two
twilight-anchored branches so each half of the night is referenced to its own −12° edge:

- **Evening** exposures vs **hours after the evening −12° twilight** (`t = 0` at dusk edge).
- **Morning** exposures vs **hours before the morning −12° twilight** (`t = 0` at dawn edge).

Both coordinates start at their twilight edge and increase toward solar midnight, and each panel
carries a secondary **top axis** translating position to **sun elevation**.  Splitting this way
avoids the double-valued fold of plotting against sun elevation directly (the sun passes each
elevation twice) and lets dusk-side and dawn-side trends be compared edge-to-edge.  The axes are
**restricted to the first 4 h** past each twilight edge, which captures the twilight-driven
regime while staying clear of solar midnight (where the two branches would overlap and the
sun-elevation mapping flattens).

Metrics (`cdb_lsstcam.visit1_quicklook`):

| Metric | Column | Units |
|---|---|---|
| PSF FWHM | `psf_sigma_median` × 2.355 × 0.2 | arcsec |
| Donut blur FWHM | `donut_blur_fwhm` | arcsec |
| AOS residual FWHM | `aos_fwhm` | arcsec |

Each twilight-anchored coordinate is derived per-exposure from the Sun's hour angle relative to
its −12° crossing (computed from the exposure's solar declination), so it is exact for each night
regardless of season.  Figures are laid out as **2 rows (evening top, morning bottom) × 3 metric
columns**; the binned **median ± 16–84 percentile band** is drawn over each branch.

Two sets of plots:
1. **All science exposures** (2026-01-01 → 2026-07-13), with baseline image-quality cuts.
2. **AOS-stability tests only** — exposures whose `scheduler_note` tags a fixed-pointing
   AOS/hexapod stare (`AOS alt:.. az:..`, `BLOCK-T698 …`, `BLOCK-T706 …`), i.e. az/el/rotation
   held constant so twilight/thermal effects are isolated from field-motion effects.

Sample: **2026-01-01 → 2026-07-13**, `img_type = 'science'`.

## Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sqlalchemy

from astropy.time import Time
from astropy.coordinates import EarthLocation, AltAz, get_sun
import astropy.units as u

%matplotlib inline

## Configuration

ConsDB access via direct PostgreSQL (pgpass creds).  Sun elevation is computed from the
exposure-midpoint timestamp at the Rubin Observatory site.

In [ ]:
# ── ConsDB / PostgreSQL ───────────────────────────────────────────────────────
PGPASS_FILE = os.path.expanduser("~/.lsst/postgres-credentials.txt")
CONSDB_HOST = "usdf-summitdb-logical-replica-svc.sdf.slac.stanford.edu"
CONSDB_DB = "exposurelog"
CONSDB_USER = "usdf"
SCHEMA = "cdb_lsstcam"


def load_pgpass(path, host, database, user):
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            parts = line.split(":")
            if len(parts) < 5:
                continue
            h, db, u_ = parts[0], parts[2], parts[3]
            pwd = ":".join(parts[4:])
            if h == host and db == database and u_ == user:
                return pwd
    raise ValueError(f"No credentials for {user}@{host}/{database}")


db_pass = load_pgpass(PGPASS_FILE, CONSDB_HOST, CONSDB_DB, CONSDB_USER)
engine = sqlalchemy.create_engine(
    f"postgresql+psycopg2://{CONSDB_USER}:{db_pass}@{CONSDB_HOST}/{CONSDB_DB}",
    connect_args={"connect_timeout": 60},
)


def consdb_query(sql):
    with engine.connect() as conn:
        return pd.read_sql_query(sqlalchemy.text(sql), conn)


# ── Sample window ─────────────────────────────────────────────────────────────
DAY_OBS_START = 20260101
DAY_OBS_END = 20260713

# ── Rubin Observatory site (Cerro Pachón) ─────────────────────────────────────
RUBIN_LOC = EarthLocation(lat=-30.2444 * u.deg, lon=-70.7494 * u.deg, height=2663 * u.m)

# ── Night definition & analysis constants ─────────────────────────────────────
TWILIGHT_ALT = -12.0  # deg — astronomical-ish twilight edge; night = sun_alt <= this
PIXEL_SCALE = 0.2  # arcsec / pixel (LSSTCam)
SIGMA_TO_FWHM = 2.355

# ── Parquet cache ─────────────────────────────────────────────────────────────
CACHE_FILE = "../data/consdb_sunalt_iq.parquet"
FORCE_REFETCH = False  # True → ignore cache, re-fetch from ConsDB

## 1. Fetch exposures + quicklook, compute sun elevation

One row per science exposure with the three IQ metrics, pointing (az/el/rotation), band,
and `scheduler_note`.  For each exposure we compute the sun altitude at the exposure midpoint
and label whether the sun is **descending** (evening) or **ascending** (morning) by probing the
altitude 10 minutes later.

In [ ]:
import pathlib as _pl

_cp = _pl.Path(CACHE_FILE)
if not FORCE_REFETCH and _cp.exists():
    df = pd.read_parquet(CACHE_FILE)
    print(f"Loaded {len(df)} exposures from cache: {CACHE_FILE}")
else:
    df = consdb_query(
        f"""
        SELECT
            e.exposure_id, e.day_obs, e.seq_num, e.obs_start, e.obs_start_mjd, e.exp_time,
            e.azimuth, e.altitude, e.sky_rotation, e.band, e.img_type, e.scheduler_note,
            q.psf_sigma_median, q.donut_blur_fwhm, q.aos_fwhm,
            q.eff_time_zero_point_scale_median
        FROM {SCHEMA}.exposure e
        LEFT JOIN {SCHEMA}.visit1_quicklook q
               ON q.day_obs = e.day_obs AND q.seq_num = e.seq_num
        WHERE e.day_obs BETWEEN {DAY_OBS_START} AND {DAY_OBS_END}
          AND e.img_type = 'science'
        ORDER BY e.obs_start_mjd
        """
    )
    df["obs_start_utc"] = pd.to_datetime(df["obs_start"], utc=True)

    # Sun altitude at exposure midpoint, and 10 min later to sign the branch.
    mjd_mid = (
        df["obs_start_mjd"].to_numpy() + (df["exp_time"].to_numpy() / 2.0) / 86400.0
    )
    t_mid = Time(mjd_mid, format="mjd", scale="utc")
    df["sun_alt"] = (
        get_sun(t_mid).transform_to(AltAz(obstime=t_mid, location=RUBIN_LOC)).alt.deg
    )

    t_next = Time(mjd_mid + 10.0 / 1440.0, format="mjd", scale="utc")
    sun_next = (
        get_sun(t_next).transform_to(AltAz(obstime=t_next, location=RUBIN_LOC)).alt.deg
    )
    df["sun_branch"] = np.where(
        sun_next < df["sun_alt"].to_numpy(), "evening", "morning"
    )

    # PSF sigma (pixels) → FWHM (arcsec).
    df["psf_fwhm_arcsec"] = df["psf_sigma_median"] * SIGMA_TO_FWHM * PIXEL_SCALE

    df.to_parquet(CACHE_FILE)
    print(f"Fetched + cached {len(df)} exposures → {CACHE_FILE}")

# ── Time relative to the twilight edges (two monotonic night-time axes) ───────
# Evening set:  hae = hours *after* the evening −12° crossing  (0 at dusk, → midnight)
# Morning set:  hbm = hours *before* the morning −12° crossing (0 at dawn, → midnight)
# Each is exact per-night (derived from the Sun's hour angle & the exposure's declination),
# so the two branches can be plotted against their own twilight-anchored clock.
#
#   • Position from solar noon:   H = LST − RA_sun  (H = 0 at noon),
#     re-centered on solar midnight → hfm (evening negative, morning positive).
#   • −12° crossing hour angle (per exposure, from its solar declination):
#       cos H_tw = (sin(−12°) − sinφ sinδ) / (cosφ cosδ),   H_tw > 0 (descending).
#     Evening crossing is at hfm_tw = H_tw − 12 (< 0); morning crossing at −hfm_tw (> 0).
#   • hae = hfm − hfm_tw ≥ 0 ;   hbm = (−hfm_tw) − hfm ≥ 0.
# Computed here (not only in the fetch branch) so a cached parquet still works.
if "hae" not in df.columns or "hbm" not in df.columns:
    _mjd_mid = (
        df["obs_start_mjd"].to_numpy() + (df["exp_time"].to_numpy() / 2.0) / 86400.0
    )
    _t = Time(_mjd_mid, format="mjd", scale="utc")
    _lst = _t.sidereal_time("apparent", longitude=RUBIN_LOC.lon)
    _sun = get_sun(_t)
    _H = (
        (_lst - _sun.ra).wrap_at(180 * u.deg).to_value(u.hourangle)
    )  # −12..12, 0 = noon
    _hfm = np.where(_H > 0, _H - 12.0, _H + 12.0)

    _phi = RUBIN_LOC.lat.to_value(u.rad)
    _dec = _sun.dec.to_value(u.rad)
    _cosH = (np.sin(np.deg2rad(TWILIGHT_ALT)) - np.sin(_phi) * np.sin(_dec)) / (
        np.cos(_phi) * np.cos(_dec)
    )
    _H_tw = (
        np.rad2deg(np.arccos(np.clip(_cosH, -1.0, 1.0))) / 15.0
    )  # h, descending crossing
    _hfm_tw = _H_tw - 12.0  # evening crossing relative to solar midnight (negative)
    df["hae"] = _hfm - _hfm_tw  # hours after evening −12° twilight
    df["hbm"] = -_hfm_tw - _hfm  # hours before morning −12° twilight

print(f"\nDate range        : day_obs {df['day_obs'].min()} → {df['day_obs'].max()}")
print(f"Sun-alt range     : {df['sun_alt'].min():.1f}° → {df['sun_alt'].max():.1f}°")
print(
    f"Hours-after-evening-twilight range : {df['hae'].min():.1f} → {df['hae'].max():.1f} h"
)
print(
    f"Hours-before-morning-twilight range: {df['hbm'].min():.1f} → {df['hbm'].max():.1f} h"
)
print(f"Branch counts     : {df['sun_branch'].value_counts().to_dict()}")
print("Metric coverage (non-null):")
for c in ["psf_fwhm_arcsec", "donut_blur_fwhm", "aos_fwhm"]:
    print(f"  {c:20s}: {df[c].notna().sum()}")

## 2. Night selection, quality cuts, and binning helpers

**Night window**: `sun_alt <= −12°` (from evening −12° down through solar midnight and back up
to morning −12°).

**Baseline image-quality cuts** (same as the sibling `ConsDB_EFD_to_PSF_Effects_Diagnosis`
notebook): drop measured `aos_fwhm >= 0.75″`, low-throughput exposures
(`eff_time_zero_point_scale_median < 0.75`), and y-band.  Missing values are *kept* (missing ≠ bad).

Binning is done in **twilight-anchored hours** — `hae` (hours after evening −12°) for the evening
branch, `hbm` (hours before morning −12°) for the morning branch.  Each branch is monotonic in its
own coordinate, so `sunalt_axis_ticks(branch, xcol)` inverts its median `sun_alt(x)` curve once to
place sun-elevation labels on the top axis (no midnight split needed).

In [ ]:
# ── Night window ──────────────────────────────────────────────────────────────
df_night = df[df["sun_alt"] <= TWILIGHT_ALT].copy()
print(f"Night exposures (sun_alt <= {TWILIGHT_ALT}°): {len(df_night)}")

# ── Baseline image-quality cuts ───────────────────────────────────────────────
IQ_CUTS = dict(aos_fwhm_max=0.75, zp_scale_min=0.75, band_exclude=["y"])


def apply_iq_cuts(d):
    keep = d["aos_fwhm"].isna() | (d["aos_fwhm"] < IQ_CUTS["aos_fwhm_max"])
    keep &= d["eff_time_zero_point_scale_median"].isna() | (
        d["eff_time_zero_point_scale_median"] >= IQ_CUTS["zp_scale_min"]
    )
    keep &= ~d["band"].isin(IQ_CUTS["band_exclude"])
    return d[keep]


df_iq = apply_iq_cuts(df_night)
print(f"After baseline IQ cuts: {len(df_iq)} ({len(df_night) - len(df_iq)} removed)")

# ── Binning in twilight-anchored hours ────────────────────────────────────────
# Restricted to the first 4 h after (before) each twilight edge: this covers the
# twilight-driven regime while staying well clear of solar midnight, where the evening
# and morning branches would otherwise overlap and the sun-elevation mapping flattens.
TWI_HOURS_MAX = 4.0
TWI_BIN_EDGES = np.arange(0.0, TWI_HOURS_MAX + 0.0001, 0.25)  # h
MIN_PER_BIN = 5


def binned_stats(d, col, xcol, edges=TWI_BIN_EDGES, min_per_bin=MIN_PER_BIN):
    """Return (centers, median, p16, p84, n) for `col` vs `xcol` in bins."""
    x = d[xcol].to_numpy()
    y = d[col].to_numpy()
    m = np.isfinite(x) & np.isfinite(y)
    x, y = x[m], y[m]
    idx = np.digitize(x, edges)
    cen, med, lo, hi, cnt = [], [], [], [], []
    for b in range(1, len(edges)):
        s = y[idx == b]
        if len(s) < min_per_bin:
            continue
        cen.append(0.5 * (edges[b - 1] + edges[b]))
        med.append(np.median(s))
        lo.append(np.percentile(s, 16))
        hi.append(np.percentile(s, 84))
        cnt.append(len(s))
    return (np.array(cen), np.array(med), np.array(lo), np.array(hi), np.array(cnt))


# ── Twilight-anchored time → sun-elevation mapping for the secondary axis ─────
# Each branch is monotonic in its own twilight-anchored coordinate (sun elevation only
# deepens from the −12° edge toward midnight), so a single inversion of the median
# sun_alt(x) curve suffices — no midnight split needed.
ELEV_TICKS = [-15, -20, -30, -40, -50, -60]  # deg


def sunalt_axis_ticks(branch, xcol, edges=TWI_BIN_EDGES):
    """Return (positions, labels) for sun-elevation ticks along a branch's time axis."""
    d = df_night[df_night["sun_branch"] == branch]
    x = d[xcol].to_numpy()
    a = d["sun_alt"].to_numpy()
    cen, alt = [], []
    for b in range(1, len(edges)):
        s = a[(x >= edges[b - 1]) & (x < edges[b])]
        if len(s) >= MIN_PER_BIN:
            cen.append(0.5 * (edges[b - 1] + edges[b]))
            alt.append(np.median(s))
    cen, alt = np.array(cen), np.array(alt)
    if len(cen) < 2:
        return [], []
    order = np.argsort(alt)  # ascending elevation for np.interp
    a_sorted, c_sorted = alt[order], cen[order]
    pos, lab = [], []
    for e in ELEV_TICKS:
        if a_sorted.min() <= e <= a_sorted.max():
            pos.append(float(np.interp(e, a_sorted, c_sorted)))
            lab.append(f"{e}")
    return pos, lab


# Metric definitions shared by both plot sets.
METRICS = [
    ("psf_fwhm_arcsec", "PSF FWHM (arcsec)", (0, 2.5)),
    ("donut_blur_fwhm", "Donut Blur FWHM (arcsec)", (0, 2.0)),
    ("aos_fwhm", "AOS Residual FWHM (arcsec)", (0, 0.8)),
]

# One row per twilight-anchored branch: (branch, x-column, x-label, colour).
BRANCH_ROWS = [
    ("evening", "hae", "Hours after evening −12° twilight", "tab:orange"),
    ("morning", "hbm", "Hours before morning −12° twilight", "tab:blue"),
]

## 3. Plotting helper

`iq_vs_time()` lays out a **2×3 grid**: metric columns (PSF FWHM, donut blur, AOS FWHM) × two
rows (evening top, morning bottom).  Each row plots its branch against its own twilight-anchored
coordinate, overlaid with the binned median line and 16–84 percentile band, and adds a **top
axis** labelling sun elevation.

In [ ]:
def _add_sunalt_top_axis(ax, branch, xcol):
    """Add a secondary top axis labelling sun elevation at the mapped time positions."""
    pos, lab = sunalt_axis_ticks(branch, xcol)
    axt = ax.twiny()
    axt.set_xlim(ax.get_xlim())
    axt.set_xticks(pos)
    axt.set_xticklabels(lab, fontsize=8)
    axt.set_xlabel("Sun elevation (deg)", fontsize=9)
    axt.tick_params(axis="x", which="both", length=3)
    return axt


# Exposures whose scheduler_note marks a near-sun twilight survey block.  These are the
# shallowest-sun (sun_alt −20°..−12°) science frames, clustered near each twilight edge.
def _twilight_near_sun_mask(d):
    return d["scheduler_note"].fillna("").str.startswith("twilight_near_sun")


TNS_COLOR = "crimson"


def iq_vs_time(
    data,
    title,
    point_alpha=0.12,
    point_size=4,
    highlight_twilight=True,
    metrics=METRICS,
):
    """2×N grid: IQ metrics (columns) vs twilight-anchored time, one branch per row.

    Top row — evening exposures vs hours *after* the evening −12° crossing.
    Bottom row — morning exposures vs hours *before* the morning −12° crossing.
    Both axes start (t = 0) at their twilight edge and increase toward solar midnight;
    a secondary top axis on each panel translates position to sun elevation.

    When ``highlight_twilight`` is set, exposures from the ``twilight_near_sun`` survey
    block are overplotted in a distinct colour (the binned median still uses all points).
    Pass ``metrics`` as a list of ``(column, ylabel, ylim)`` tuples to override the
    default three IQ columns (e.g. a raw-vs-corrected PSF comparison).
    """
    fig, axes = plt.subplots(
        2, len(metrics), figsize=(6.3 * len(metrics), 10.5), squeeze=False
    )
    xlim = (TWI_BIN_EDGES[0], TWI_BIN_EDGES[-1])
    for r, (branch, xcol, xlabel, color) in enumerate(BRANCH_ROWS):
        sub_branch = data[data["sun_branch"] == branch]
        for c, (col, ylabel, ylim) in enumerate(metrics):
            ax = axes[r, c]
            sub = sub_branch.dropna(subset=[col])
            is_tns = _twilight_near_sun_mask(sub) if highlight_twilight else False
            base = sub[~is_tns] if highlight_twilight else sub
            ax.scatter(
                base[xcol],
                base[col],
                s=point_size,
                alpha=point_alpha,
                color=color,
                edgecolors="none",
                rasterized=True,
                label=f"{branch}  (n={len(base)})",
            )
            if highlight_twilight:
                tns = sub[is_tns]
                if len(tns):
                    ax.scatter(
                        tns[xcol],
                        tns[col],
                        s=point_size * 4,
                        alpha=0.75,
                        color=TNS_COLOR,
                        edgecolors="black",
                        linewidths=0.3,
                        rasterized=True,
                        zorder=5,
                        label=f"twilight_near_sun  (n={len(tns)})",
                    )
            cen, med, lo, hi, cnt = binned_stats(sub, col, xcol)
            if len(cen):
                ax.plot(cen, med, "-", color="black", lw=2.2, label="binned median")
                ax.fill_between(
                    cen, lo, hi, color="gray", alpha=0.25, label="16–84 pct"
                )
            ax.set_xlabel(xlabel)
            ax.set_ylabel(ylabel)
            ax.set_ylim(*ylim)
            ax.set_xlim(*xlim)
            ax.grid(True, alpha=0.3)
            ax.legend(loc="upper center", fontsize=7.5, framealpha=0.9)
            _add_sunalt_top_axis(ax, branch, xcol)
    fig.suptitle(title, fontsize=14, y=1.005)
    fig.tight_layout()
    return fig, axes

## 4. All science exposures

Full-night image quality for every science exposure passing the baseline quality cuts, split by
branch.  Top row: evening exposures measured forward from the evening −12° edge.  Bottom row:
morning exposures measured backward from the morning −12° edge.  Read elapsed time on the bottom
axis and the corresponding sun elevation on the top.  Both branches degrade toward their twilight
edge (`t → 0`) as sky background and near-sun conditions rise.

Exposures from the **`twilight_near_sun`** survey block (the shallowest-sun science frames, sun_alt −20°..−12°, clustered against each twilight edge) are **highlighted in crimson**.

In [ ]:
fig, axes = iq_vs_time(
    df_iq,
    f"Image Quality vs Time of Night — all science exposures "
    f"({DAY_OBS_START} → {DAY_OBS_END}, n={len(df_iq)})",
)
plt.show()

### 4a. All science exposures — `twilight_near_sun` excluded

The same 2×3 layout, but with the **`twilight_near_sun`** survey-block exposures removed from the analysis entirely (dropped before both the scatter and the binned median). These are the shallowest-sun frames (sun_alt −20°..−12°), taken deliberately close to the Sun at each twilight edge, so they can skew the near-edge (`t → 0`) trend. Excluding them isolates the image-quality behaviour of the ordinary survey cadence.

In [ ]:
# Drop the twilight_near_sun survey block, then re-run the standard plot.
df_iq_no_tns = df_iq[~_twilight_near_sun_mask(df_iq)].copy()
n_removed = len(df_iq) - len(df_iq_no_tns)
print(
    f"Excluded {n_removed} twilight_near_sun exposures "
    f"({df_iq_no_tns.sun_branch.value_counts().to_dict()} remaining by branch)"
)

fig, axes = iq_vs_time(
    df_iq_no_tns,
    f"Image Quality vs Time of Night — all science exposures, "
    f"twilight_near_sun excluded ({DAY_OBS_START} → {DAY_OBS_END}, n={len(df_iq_no_tns)})",
    highlight_twilight=False,
)
plt.show()

### 4b. PSF FWHM corrected to 500 nm and zenith

If the residual morning upturn is a **low-elevation (high-airmass) bias** rather than a genuine
twilight effect, normalizing the PSF for airmass and wavelength should flatten it.  Under
atmosphere-dominated (Kolmogorov / von Kármán) seeing, delivered FWHM scales as

$$\mathrm{FWHM} \;\propto\; \lambda^{-1/5}\; X^{\,3/5},\qquad X=\sec z = 1/\sin(\mathrm{alt}),$$

so the seeing referred to **500 nm at zenith** is

$$\mathrm{FWHM}_{500,\,\mathrm{zen}} \;=\; \mathrm{FWHM}_{\mathrm{obs}}\;
\left(\frac{\lambda_{\mathrm{eff}}}{500\ \mathrm{nm}}\right)^{1/5}\; X^{-3/5}.$$

- **Wavelength** term uses each exposure's band effective wavelength
  (u 367, g 483, r 622, i 754, z 869 nm — y is already excluded by the baseline cuts).
- **Airmass** term uses the pointing altitude (`altitude`), $X=1/\sin(\mathrm{alt})$.

This is the same normalization ConsDB stores as `seeing_zenith_500nm`; we compute it inline from
`psf_sigma_median` so the correction is explicit.  The `twilight_near_sun` block remains excluded.
Left column: raw PSF FWHM; right column: corrected.  If the correction is the whole story, the
right-column morning curve should be flat toward `t → 0`.

In [ ]:
# ── PSF FWHM corrected to 500 nm, zenith (Kolmogorov: λ^-1/5, X^3/5) ──────────
# Band effective wavelengths (nm), LSST throughput-weighted.
BAND_LAMBDA_NM = {
    "u": 367.0,
    "g": 483.0,
    "r": 622.0,
    "i": 754.0,
    "z": 869.0,
    "y": 971.0,
}
LAMBDA_REF_NM = 500.0


def correct_psf_to_500nm_zenith(d):
    """FWHM_obs → FWHM at 500 nm & zenith:  × (λ_eff/500)^(1/5) × X^(-3/5)."""
    lam = d["band"].map(BAND_LAMBDA_NM).to_numpy(dtype=float)
    airmass = 1.0 / np.sin(np.deg2rad(d["altitude"].to_numpy(dtype=float)))
    wave_term = (
        lam / LAMBDA_REF_NM
    ) ** 0.2  # λ^(-1/5), applied to bluer-ref → factor >? see note
    airmass_term = airmass ** (-0.6)  # X^(-3/5)
    return d["psf_fwhm_arcsec"].to_numpy(dtype=float) * wave_term * airmass_term


# NOTE on the wavelength sign: seeing ∝ λ^(-1/5), so FWHM(500) = FWHM(λ) · (500/λ)^(-1/5)
#   = FWHM(λ) · (λ/500)^(1/5).  Redder bands (λ>500) have *smaller* delivered FWHM, so scaling
#   them to 500 nm *increases* the value → (λ/500)^(1/5) > 1.  This is the correct direction.

df_corr = df_iq_no_tns.copy()
df_corr["psf_fwhm_500zen"] = correct_psf_to_500nm_zenith(df_corr)

# Sanity: median raw vs corrected, and airmass span.
_am = 1.0 / np.sin(np.deg2rad(df_corr["altitude"]))
print(
    f"Airmass X = sec(z):  median {_am.median():.3f},  95th pct {_am.quantile(0.95):.3f}"
)
print(
    f"PSF FWHM median  raw {df_corr['psf_fwhm_arcsec'].median():.3f}\"  →  "
    f"corrected {df_corr['psf_fwhm_500zen'].median():.3f}\""
)

# Two-column layout: raw PSF (left) vs corrected PSF (right), evening/morning rows.
PSF_METRICS = [
    ("psf_fwhm_arcsec", "PSF FWHM — raw (arcsec)", (0, 2.5)),
    ("psf_fwhm_500zen", "PSF FWHM — 500 nm, zenith (arcsec)", (0, 2.5)),
]

fig, axes = iq_vs_time(
    df_corr,
    "PSF FWHM raw vs corrected to 500 nm & zenith — twilight_near_sun excluded "
    f"(n={len(df_corr)})",
    highlight_twilight=False,
    metrics=PSF_METRICS,
)
plt.show()

## 5. AOS-stability tests (fixed az / el / rotation)

Exposures whose `scheduler_note` marks a fixed-pointing AOS/hexapod stability stare.  In these
blocks the telescope azimuth, altitude, and rotation are held constant (see verification below),
so field-motion and airmass effects are removed and any sun-elevation trend reflects
twilight/thermal conditioning of the optics rather than pointing changes.

Tagged families:
- `AOS alt:.. az:..` — AOS stares
- `BLOCK-T698 …`, `BLOCK-T706 …`, `BLOCK-T703 …` — AOS/hexapod engineering blocks (fixed alt/az/rotTel)

In [ ]:
note = df_night["scheduler_note"].fillna("")
AOS_PREFIXES = ("AOS ", "BLOCK-T698", "BLOCK-T706", "BLOCK-T703")
is_aos = note.str.startswith(AOS_PREFIXES)

df_aos_night = df_night[is_aos].copy()
df_aos = apply_iq_cuts(df_aos_night)

print(f"AOS-stability night exposures      : {len(df_aos_night)}")
print(f"  after baseline IQ cuts           : {len(df_aos)}")
print(
    f"  branch counts                    : {df_aos['sun_branch'].value_counts().to_dict()}"
)


# Verify pointing is actually held fixed within each note (azimuth std computed on a
# wrap-safe basis, since az clusters near 0°/360° for the az:0 stares).
def _circ_std_deg(a):
    r = np.deg2rad(a.to_numpy())
    C, S = np.cos(r).mean(), np.sin(r).mean()
    R = np.hypot(C, S)
    return np.rad2deg(np.sqrt(-2.0 * np.log(max(R, 1e-12))))


stab = (
    df_aos_night.groupby("scheduler_note")
    .agg(
        n=("exposure_id", "size"),
        alt_std=("altitude", "std"),
        rot_std=("sky_rotation", "std"),
    )
    .sort_values("n", ascending=False)
)
stab["az_circstd"] = df_aos_night.groupby("scheduler_note")["azimuth"].apply(
    _circ_std_deg
)
print("\nPointing stability within top AOS-stability notes (deg):")
print(stab.head(10).to_string(float_format=lambda v: f"{v:.3f}"))

In [ ]:
fig, axes = iq_vs_time(
    df_aos,
    f"Image Quality vs Time of Night — AOS-stability tests (fixed az/el/rot), n={len(df_aos)}",
    point_alpha=0.30,
    point_size=10,
    highlight_twilight=False,
)
plt.show()

### AOS-stability tests, coloured by fixed pointing

Because each AOS/BLOCK note corresponds to a distinct fixed (alt, az) — and the metrics may
differ between pointings — this panel colours the AOS-stability points by their nominal pointing
group rather than by twilight branch, to check whether any sun-elevation trend is pointing-driven.

In [ ]:
import re


def pointing_key(n):
    """Extract a compact 'alt.. az..' label from an AOS/BLOCK scheduler_note."""
    m_alt = re.search(r"alt:(-?\d+\.?\d*)", n)
    m_az = re.search(r"az:(-?\d+\.?\d*)", n)
    alt = m_alt.group(1) if m_alt else "?"
    az = m_az.group(1) if m_az else "?"
    return f"alt {alt} / az {az}"


df_aos = df_aos.copy()
df_aos["pointing"] = df_aos["scheduler_note"].fillna("").map(pointing_key)
top_pointings = df_aos["pointing"].value_counts().head(6).index.tolist()

fig, axes = plt.subplots(2, 3, figsize=(19, 10.5))
xlim = (TWI_BIN_EDGES[0], TWI_BIN_EDGES[-1])
cmap = plt.get_cmap("tab10")
for r, (branch, xcol, xlabel, _color) in enumerate(BRANCH_ROWS):
    sub_branch = df_aos[df_aos["sun_branch"] == branch]
    for c, (col, ylabel, ylim) in enumerate(METRICS):
        ax = axes[r, c]
        for k, pk in enumerate(top_pointings):
            sub = sub_branch[sub_branch["pointing"] == pk].dropna(subset=[col])
            if len(sub) == 0:
                continue
            ax.scatter(
                sub[xcol],
                sub[col],
                s=10,
                alpha=0.35,
                color=cmap(k),
                edgecolors="none",
                label=f"{pk} (n={len(sub)})",
            )
        ax.set_xlabel(xlabel)
        ax.set_ylabel(f"{ylabel}  [{branch}]")
        ax.set_ylim(*ylim)
        ax.set_xlim(*xlim)
        ax.grid(True, alpha=0.3)
        ax.legend(loc="upper center", fontsize=6.5, framealpha=0.9)
        _add_sunalt_top_axis(ax, branch, xcol)
fig.suptitle(
    "AOS-stability tests vs Time of Night — coloured by fixed pointing "
    "(top: evening, bottom: morning)",
    fontsize=14,
    y=1.005,
)
fig.tight_layout()
plt.show()

## Notes

- **Two twilight-anchored axes**, one per branch: `hae` = hours *after* the evening −12° crossing
  (0 at dusk edge), `hbm` = hours *before* the morning −12° crossing (0 at dawn edge).  Both
  increase toward solar midnight.  Each is computed per-exposure from the Sun's hour angle
  relative to its −12° crossing (`cos H_tw = (sin(−12°) − sinφ sinδ)/(cosφ cosδ)`), using the
  exposure's solar declination — exact per night, no clock offset.
- **Top axis** translates position to **sun elevation** by inverting each branch's median
  `sun_alt(x)` curve (`sunalt_axis_ticks(branch, xcol)`).  Because the season-averaged geometry
  varies slightly night-to-night, the elevation ticks are representative medians, not exact for
  any single night.
- **PSF FWHM** = `psf_sigma_median × 2.355 × 0.2 arcsec/pix`.  **Donut blur** and **AOS FWHM**
  are already in arcsec in `visit1_quicklook`.
- Baseline quality cuts (`aos_fwhm < 0.75″`, `zp_scale ≥ 0.75`, drop y-band) match the sibling
  `optical_psf/ConsDB_EFD_to_PSF_Effects_Diagnosis.ipynb`.
- **AOS-stability tests** are identified by `scheduler_note` prefix (`AOS …`, `BLOCK-T698/706/703 …`);
  the printed table verifies altitude/rotation std ≈ 0 and wrap-safe azimuth circular-std ≈ 0
  within each note, confirming fixed pointing.
- Data cached at `../data/consdb_sunalt_iq.parquet`; set `FORCE_REFETCH = True` to rebuild.